# Notebook 1 · Pre-tratamiento de los datos
**Proyecto de fin de curso – Visualización de datos (UNAP)**
**Consumo de energía eléctrica de los clientes de Electro Puno S.A.A.**

Fuente: [Portal de Datos Abiertos del Perú](https://www.datosabiertos.gob.pe/dataset/consumo-de-energ%C3%ADa-el%C3%A9ctrica-de-los-clientes-de-electro-puno-saa)

Etapa KDD: **selección → pre-procesamiento → transformación**. Objetivos:
1. Cargar **todos** los archivos mensuales de `data/raw/` (CSV y XLSX, ene-2023 → ago-2026) y llevarlos a un solo formato.
2. Dar formato correcto a los tipos (numéricos, fechas, periodo comercial).
3. Limpiar textos (`NU¥OA` → `NUÑOA`) y corregir ubigeos inconsistentes.
4. **Geo-localizar** cada distrito (ubigeo INEI, centroide, área) con un GeoJSON complementario.
5. Agregar columnas derivadas (nivel de tensión, grupo tarifario, indicadores unitarios).
6. Guardar una estructura limpia (`data/processed/`) para el EDA y el dashboard.

> **Cómo cargar más meses:** descarga cada archivo del portal y guárdalo en `data/raw/` con el número correlativo
> delante (p. ej. `54 ReporteDatosAbiertosLibro_202609.csv`). El notebook lee **todos** los CSV/XLSX de esa carpeta;
> si la lista de archivos cambia, la caché se reconstruye sola.

## 1. Librerías y rutas

In [2]:
import importlib.util
import io
import json
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW = Path("data/raw")
GEO = Path("data/geo/peru_distrital_simple.geojson")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)
CACHE, CACHE_LOG = OUT / "raw_unificado.parquet", OUT / "raw_archivos.csv"

In [3]:
import sys

print(sys.executable)

d:\visualizacion_data_electropuno\.venv\Scripts\python.exe


## 2. Carga de datos
Cada archivo del portal corresponde a un periodo comercial (mes), pero el formato **cambia** entre archivos:

| Aspecto | Variantes encontradas |
|---|---|
| Nivel de detalle | 2023-2025: **una fila por suministro** · 2026: filas **ya agregadas** con la columna `Clientes` |
| Nombres de columna | `CONSUMO`/`CEA`, `FACTURACIÓN`/`M_Total`, `PERIODO`/`CodigoPeriodoComercial`… |
| Separador | `;` (casi todos) o `,` |
| Codificación | `utf-8` (con o sin BOM) o `cp1252` |
| Cabecera | `12 Octubre ELPU 202310.csv` no la tiene |
| Formato | `30 ReporteDatosAbiertos 202412.xlsx` es Excel |

Por eso las columnas se asignan **por posición** (el orden es el mismo en todos los archivos) y cada archivo se lleva al
formato agregado de 2026: se suman `Clientes`, `CEA` y `M_Total` por ubigeo, distrito, tarifa, periodo y estado.

El mes se toma de la columna de periodo (**no del nombre del archivo**): varios archivos repiten un mes ya cargado
(p. ej. `19 …202406.csv` contiene en realidad 202402) y se descartan; quedan anotados en la tabla `log`.
Solo se conservan los registros del departamento de **Puno** (los archivos de 2023 traen 1 suministro MT4 de Lima).
El resultado se guarda en caché (`raw_unificado.parquet`) para que las siguientes ejecuciones tarden segundos.
Si modificas `leer_archivo`, borra `data/processed/raw_unificado.parquet` para volver a procesar.

In [4]:
# Posición de las columnas en cada formato (los nombres cambian entre archivos, el orden no)
COLS_SUMINISTRO = ["Codigo", "CodigoUbigeo", "NombreDepartamento", "NombreProvincia", "NombreDistrito", "InicioContrato",
                   "Tarifa", "CodigoPeriodoComercial", "CEA", "M_Total", "NombreEstadoSuministro", "Fecha"]
COLS_AGREGADO = ["Codigo", "CodigoUbigeo", "NombreDepartamento", "NombreProvincia", "NombreDistrito",
                 "Tarifa", "CodigoPeriodoComercial", "Clientes", "CEA", "M_Total", "NombreEstadoSuministro", "Fecha"]
CLAVES = ["CodigoUbigeo", "NombreDepartamento", "NombreProvincia", "NombreDistrito", "Tarifa",
          "CodigoPeriodoComercial", "NombreEstadoSuministro", "Fecha"]


def a_numero(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.strip().str.replace(",", "", regex=False), errors="coerce").fillna(0.0)


def a_fecha(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    f = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return f.fillna(pd.to_datetime(s, format="mixed", dayfirst=True, errors="coerce"))   # el XLSX trae fechas ISO


def leer_archivo(ruta: Path) -> tuple[pd.DataFrame, str]:
    # Devuelve el archivo en el formato agregado de 2026, sin importar separador, codificación, cabecera o csv/xlsx
    if ruta.suffix.lower() == ".xlsx":
        motor = "calamine" if importlib.util.find_spec("python_calamine") else "openpyxl"
        crudo = pd.read_excel(ruta, engine=motor, dtype=str)
        cabecera = " ".join(map(str, crudo.columns))
    else:
        b = ruta.read_bytes()
        try:
            txt = b.decode("utf-8-sig")
        except UnicodeDecodeError:
            txt = b.decode("cp1252")
        primera = txt.split("\n", 1)[0]
        sep = ";" if primera.count(";") >= primera.count(",") else ","
        sin_cabecera = primera.split(sep)[0].strip().isdigit()
        cabecera = "" if sin_cabecera else primera
        crudo = pd.read_csv(io.StringIO(txt), sep=sep, header=None if sin_cabecera else 0, dtype=str,
                            usecols=range(12), keep_default_na=False)
    agregado = "CLIENTES" in cabecera.upper()
    crudo = crudo.iloc[:, :12].copy()
    crudo.columns = COLS_AGREGADO if agregado else COLS_SUMINISTRO

    for c in ["CodigoUbigeo", "NombreDepartamento", "NombreProvincia", "NombreDistrito", "Tarifa", "NombreEstadoSuministro"]:
        crudo[c] = crudo[c].astype(str).str.strip()
    crudo["CodigoUbigeo"] = crudo["CodigoUbigeo"].str.replace(r"\.0$", "", regex=True)
    crudo["CodigoPeriodoComercial"] = pd.to_numeric(crudo["CodigoPeriodoComercial"], errors="coerce")
    crudo = crudo[crudo["CodigoPeriodoComercial"].notna()]
    crudo["CodigoPeriodoComercial"] = crudo["CodigoPeriodoComercial"].astype(int)
    crudo["Clientes"] = a_numero(crudo["Clientes"]) if agregado else 1.0
    crudo["CEA"], crudo["M_Total"] = a_numero(crudo["CEA"]), a_numero(crudo["M_Total"])
    crudo["Fecha"] = a_fecha(crudo["Fecha"]).dt.normalize()
    crudo = crudo[crudo["NombreDepartamento"].str.upper() == "PUNO"]   # 2023: 1 suministro MT4 en Santiago de Surco (Lima)

    df = crudo.groupby(CLAVES, as_index=False, dropna=False)[["Clientes", "CEA", "M_Total"]].sum()
    df["Clientes"] = df["Clientes"].round().astype(int)
    df.insert(0, "Codigo", np.arange(1, len(df) + 1))
    df["archivo"] = ruta.name
    return df, "agregado (Clientes)" if agregado else "por suministro"


def numero_archivo(ruta: Path) -> int:
    m = re.match(r"\s*(\d+)", ruta.name)
    return int(m.group(1)) if m else 10**6


def construir_cache(archivos: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    vistos, partes, log = set(), [], []
    for ruta in archivos:
        t = time.time()
        d, formato = leer_archivo(ruta)
        pers = sorted(d["CodigoPeriodoComercial"].unique())
        nuevos = [p for p in pers if p not in vistos]
        estado = "usado" if nuevos == pers else ("duplicado (periodo ya cargado)" if not nuevos else "usado parcialmente")
        log.append({"archivo": ruta.name, "bytes": ruta.stat().st_size, "formato": formato,
                    "periodo": ", ".join(map(str, pers)), "filas": len(d), "clientes": int(d["Clientes"].sum()),
                    "estado": estado})
        print(f"{ruta.name:45s} → {log[-1]['periodo']:8s} {len(d):>6,} filas · {estado} ({time.time() - t:.1f}s)")
        if nuevos:
            partes.append(d[d["CodigoPeriodoComercial"].isin(nuevos)])
            vistos.update(nuevos)
    raw = pd.concat(partes, ignore_index=True)
    log = pd.DataFrame(log)
    raw.to_parquet(CACHE, index=False)
    log.to_csv(CACHE_LOG, index=False, encoding="utf-8-sig")
    return raw, log


archivos = sorted([*RAW.glob("*.csv"), *RAW.glob("*.xlsx")], key=lambda p: (numero_archivo(p), p.name))
print(f"Archivos encontrados en {RAW}: {len(archivos)}")

firma = {(a.name, a.stat().st_size) for a in archivos}
if CACHE.exists() and CACHE_LOG.exists() and set(pd.read_csv(CACHE_LOG)[["archivo", "bytes"]].itertuples(index=False, name=None)) == firma:
    raw, log = pd.read_parquet(CACHE), pd.read_csv(CACHE_LOG)
    print("Datos leídos de la caché (se reconstruye sola si cambian los archivos de data/raw).")
else:
    raw, log = construir_cache(archivos)

print("Dimensión:", raw.shape)
raw.head()

Archivos encontrados en data\raw: 43
Datos Abiertos Mayo 2023.csv                  → 202305      652 filas · usado (3.7s)
Datos Abiertos Septiembre.csv                 → 202309      657 filas · usado (3.2s)
Datos abiertos Agosto.csv                     → 202308      657 filas · usado (2.6s)
DatosAbiertos_ Enero2023.csv                  → 202301      651 filas · usado (2.5s)
DatosAbiertos_ Marzo2023.csv                  → 202303      653 filas · usado (2.8s)
DatosAbiertos_Abril2023.csv                   → 202304      653 filas · usado (3.5s)
DatosAbiertos_Febrero2023.csv                 → 202302      652 filas · usado (3.1s)
Octubre ELPU 202310.csv                       → 202310      658 filas · usado (3.3s)
Reporte Datos Abiertos 02-2025_.csv           → 202506      668 filas · usado (3.0s)
Reporte Datos Abiertos 03-2025.csv            → 202503      678 filas · usado (3.0s)
Reporte Datos Abiertos 04-2025.csv            → 202504      682 filas · usado (2.9s)
Reporte Datos Abiertos 06-20

ImportError: `Import openpyxl` failed.  Use pip, conda, or your preferred package management tool to install the openpyxl package.

### 2.1 Archivos revisados y cobertura temporal

In [5]:
periodos = sorted(raw["CodigoPeriodoComercial"].unique())
rango = pd.period_range(pd.Period(str(periodos[0]), "M"), pd.Period(str(periodos[-1]), "M"), freq="M")
faltan = [p.strftime("%Y-%m") for p in rango if int(p.strftime("%Y%m")) not in set(periodos)]
print(f"{len(log)} archivos · {(log['estado'] != 'duplicado (periodo ya cargado)').sum()} usados · "
      f"{len(periodos)} meses ({periodos[0]} → {periodos[-1]})")
print("Meses sin archivo propio:", faltan or "ninguno")
log

52 archivos · 42 usados · 42 meses (202301 → 202608)
Meses sin archivo propio: ['2024-06', '2025-02']


,archivo,bytes,formato,periodo,filas,clientes,estado
0,1 DatosAbiertos_ Enero2023.csv,28739700,por suministro,202301,651,336735,usado
1,4 DatosAbiertos_Febrero2023.csv,28716651,por suministro,202302,652,336783,usado
2,5 DatosAbiertos_ Marzo2023.csv,28631343,por suministro,202303,653,336987,usado
3,6 DatosAbiertos_Abril2023.csv,28682462,por suministro,202304,653,337449,usado
4,7 Datos Abiertos Mayo 2023.csv,28306769,por suministro,202305,652,338473,usado
5,8 ReporteDatosAbiertos_202306.csv,28517724,por suministro,202306,653,339870,usado
6,9 ReporteDatosAbiertos_202307.csv,28578628,por suministro,202307,656,340632,usado
7,10 Datos abiertos Agosto.csv,28628169,por suministro,202308,657,341197,usado
8,11 Datos Abiertos Septiembre.csv,28664769,por suministro,202309,657,341641,usado
9,12 Octubre ELPU 202310.csv,28723596,por suministro,202310,658,342268,usado


### 2.2 Diccionario de datos
| Columna | Descripción |
|---|---|
| `Codigo` | Correlativo del registro dentro del archivo (reasignado tras unificar) |
| `CodigoUbigeo` | Ubigeo (INEI) asignado al registro |
| `NombreDepartamento / Provincia / Distrito` | Ubicación geográfica |
| `Tarifa` | Opción tarifaria (BT = baja tensión, MT = media tensión) |
| `CodigoPeriodoComercial` | Periodo facturado `AAAAMM` |
| `Clientes` | Número de suministros agrupados en el registro |
| `CEA` | Consumo de energía activa (kWh) |
| `M_Total` | Monto total facturado (S/) |
| `NombreEstadoSuministro` | Estado del suministro |
| `Fecha` | Fecha de corte/publicación del reporte |
| `archivo` | Archivo de `data/raw/` del que proviene el registro |

**Unidad de observación:** cada fila agrupa a *n* clientes de un mismo ubigeo, distrito, tarifa, estado y periodo
(no es un cliente individual). Los archivos 2023-2025, que vienen por suministro, se agregan a este mismo nivel.

In [6]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 27875 entries, 0 to 27874
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Codigo                  27875 non-null  int32         
 1   CodigoUbigeo            27875 non-null  str           
 2   NombreDepartamento      27875 non-null  str           
 3   NombreProvincia         27875 non-null  str           
 4   NombreDistrito          27875 non-null  str           
 5   Tarifa                  27875 non-null  str           
 6   CodigoPeriodoComercial  27875 non-null  int32         
 7   NombreEstadoSuministro  27875 non-null  str           
 8   Fecha                   27875 non-null  datetime64[ns]
 9   Clientes                27875 non-null  int32         
 10  CEA                     27875 non-null  float64       
 11  M_Total                 27875 non-null  float64       
 12  archivo                 27875 non-null  str           
dt

## 3. Calidad de datos

In [7]:
calidad = pd.DataFrame({
    "tipo": raw.dtypes.astype(str),
    "nulos": raw.isna().sum(),
    "%nulos": raw.isna().mean() * 100,
    "valores_unicos": raw.nunique(),
})
calidad

,tipo,nulos,%nulos,valores_unicos
Codigo,int32,0,0.00,697
CodigoUbigeo,str,0,0.00,112
NombreDepartamento,str,0,0.00,1
NombreProvincia,str,0,0.00,13
NombreDistrito,str,0,0.00,108
Tarifa,str,0,0.00,10
CodigoPeriodoComercial,int32,0,0.00,42
NombreEstadoSuministro,str,0,0.00,1
Fecha,datetime64[ns],0,0.00,32
Clientes,int32,0,0.00,1950


In [8]:
print("Filas duplicadas (todas las columnas):", raw.duplicated().sum())
print("Duplicados por (archivo, Codigo):", raw.duplicated(["archivo", "Codigo"]).sum())
print("Valores negativos:", (raw[["Clientes", "CEA", "M_Total"]] < 0).sum().to_dict())
print("Registros con CEA = 0:", (raw["CEA"] == 0).sum())
print("Registros con M_Total = 0:", (raw["M_Total"] == 0).sum())
print("Estados de suministro:", raw["NombreEstadoSuministro"].unique())

Filas duplicadas (todas las columnas): 0
Duplicados por (archivo, Codigo): 0
Valores negativos: {'Clientes': 0, 'CEA': 0, 'M_Total': 29}
Registros con CEA = 0: 7004
Registros con M_Total = 0: 229
Estados de suministro: <ArrowStringArray>
['NORMAL']
Length: 1, dtype: str


## 4. Limpieza y formato
### 4.1 Textos
El archivo trae la `Ñ` mal codificada como `¥` (`NU¥OA`, `MU¥ANI`). Se corrige y se normalizan espacios/mayúsculas.

In [9]:
def limpiar_texto(s: pd.Series) -> pd.Series:
    return s.astype(str).str.replace("¥", "Ñ", regex=False).str.strip().str.upper()

df = raw.copy()
for c in ["NombreDepartamento", "NombreProvincia", "NombreDistrito", "Tarifa", "NombreEstadoSuministro"]:
    df[c] = limpiar_texto(df[c])

df = df.rename(columns={
    "CodigoUbigeo": "ubigeo_origen", "NombreDepartamento": "departamento",
    "NombreProvincia": "provincia", "NombreDistrito": "distrito", "Tarifa": "tarifa",
    "CodigoPeriodoComercial": "periodo_cod", "Clientes": "clientes", "CEA": "consumo_kwh",
    "M_Total": "monto_soles", "NombreEstadoSuministro": "estado", "Fecha": "fecha_reporte",
    "Codigo": "codigo",
})
sorted(df.loc[df["distrito"].str.contains("Ñ"), "distrito"].unique())

['MAÑAZO', 'MUÑANI', 'NUÑOA']

### 4.2 Fechas y periodo comercial (dimensión temporal)

In [10]:
df["periodo"] = pd.to_datetime(df["periodo_cod"].astype(str), format="%Y%m")
df["anio"] = df["periodo"].dt.year
df["mes"] = df["periodo"].dt.month
df["fecha_reporte"] = pd.to_datetime(df["fecha_reporte"], errors="coerce")   # ya viene como fecha desde la carga
df[["periodo_cod", "periodo", "fecha_reporte"]].drop_duplicates().sort_values("periodo_cod").reset_index(drop=True)

,periodo_cod,periodo,fecha_reporte
0,202301,2023-01-01,2023-05-29
1,202302,2023-02-01,2023-05-29
2,202303,2023-03-01,2023-05-29
3,202304,2023-04-01,2023-05-29
4,202305,2023-05-01,2023-06-26
5,202306,2023-06-01,2023-07-19
6,202307,2023-07-01,2023-08-31
7,202308,2023-08-01,2023-09-28
8,202309,2023-09-01,2023-10-17
9,202310,2023-10-01,2023-11-30


### 4.3 Tarifas: nivel de tensión y grupo tarifario
Se agregan dos columnas para facilitar el análisis:
- **nivel_tension**: `BT` (baja) o `MT` (media), según el prefijo de la tarifa.
- **grupo_tarifa**: agrupación de opciones tarifarias similares.

⚠️ En `MT2`/`MT3` (y algunas BT con medición de potencia) el campo `CEA` suele venir en 0 aunque sí hay facturación:
la energía se registra en otros conceptos (horas punta/fuera de punta). Por eso los indicadores de kWh se interpretan
principalmente para baja tensión.

In [11]:
df["nivel_tension"] = np.where(df["tarifa"].str.startswith("MT"), "MT", "BT")
grupos = {
    "BT5B": "BT5B residencial", "BT5A": "BT5 otras", "BT5D": "BT5 otras",
    "BT6": "BT potencia", "BT2": "BT potencia", "BT3": "BT potencia", "BT4": "BT potencia",
    "MT2": "Media tensión", "MT3": "Media tensión", "MT4": "Media tensión",
}
df["grupo_tarifa"] = df["tarifa"].map(grupos).fillna("Otras")
pd.crosstab(df["tarifa"], df["nivel_tension"], values=df["clientes"], aggfunc="sum", margins=True)

nivel_tension,BT,MT,All
tarifa,,,
BT2,125.00,NaN,125
BT3,674.00,NaN,674
BT4,"1,765.00",NaN,1765
BT5A,577.00,NaN,577
BT5B,"14,529,237.00",NaN,14529237
BT5D,"82,899.00",NaN,82899
BT6,"12,448.00",NaN,12448
MT2,NaN,"12,635.00",12635
MT3,NaN,"8,189.00",8189


### 4.4 Corrección de ubigeos
El `CodigoUbigeo` no siempre coincide con el distrito declarado. Por ejemplo, hay filas con ubigeo `210101` (Puno)
y distrito `JULIACA`. Se toma el **nombre de provincia + distrito** como fuente de verdad y se le asigna el ubigeo
oficial del INEI a partir del GeoJSON distrital (el mismo de la sesión 11).

In [12]:
inconsistentes = (df.groupby(["provincia", "distrito"])["ubigeo_origen"].nunique() > 1).sum()
print("Pares provincia-distrito con más de un ubigeo en origen:", inconsistentes)
df.groupby(["provincia", "distrito"])["ubigeo_origen"].unique().head(10)

Pares provincia-distrito con más de un ubigeo en origen: 99


provincia  distrito                 
AZANGARO   ACHAYA                               [210202, 210205, 210605, 210703, 211101]
           ARAPA                                        [210201, 210203, 210206, 210708]
           ASILLO                                                       [210201, 210204]
           AZANGARO                     [210101, 210201, 210204, 210214, 210708, 211101]
           CAMINACA                                             [210205, 211101, 210703]
           CHUPA                                                        [210203, 210206]
           JOSE DOMINGO CHOQUEHUANCA                            [210201, 210207, 210214]
           MUÑANI                                       [210208, 211001, 211003, 210204]
           POTONI                                               [210209, 210306, 210701]
           SAMAN                                                                [210210]
Name: ubigeo_origen, dtype: object

## 5. Geo-localización (datos complementarios)
Se usa `peru_distrital_simple.geojson` (límites distritales INEI). Para cada distrito de Puno se obtiene:
ubigeo oficial (`IDDIST`), **centroide** (lat/lon) y **área** en km² (`AREA_MINAM` está en hectáreas).

Casos especiales detectados:
- `CAPASO` se escribe `CAPAZO` en el GeoJSON.
- `PUTINA PUNCU` es `SAN PEDRO DE PUTINA PUNCO`.
- `AMANTANI` no tiene geometría en el archivo: se le asigna un centroide manual (no aparece en el coroplético).
- `SAN MIGUEL` (San Román) se creó en 2016 a partir de Juliaca y **no tiene polígono propio** en el GeoJSON:
  conserva su ubigeo `211105` pero en el mapa se dibuja sobre el polígono de Juliaca (`geo_id = 211101`).

In [13]:
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s).replace("¥", "Ñ"))
    return "".join(c for c in s if not unicodedata.combining(c)).upper().strip()

def centroide(geom: dict) -> tuple:
    """Centroide ponderado por área (fórmula del polígono) del anillo exterior más grande."""
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    ring = max((p[0] for p in polys), key=len)
    x, y = np.array(ring)[:, 0], np.array(ring)[:, 1]
    a = x[:-1] * y[1:] - x[1:] * y[:-1]
    A = a.sum() / 2
    if A == 0:
        return float(y.mean()), float(x.mean())
    cx = ((x[:-1] + x[1:]) * a).sum() / (6 * A)
    cy = ((y[:-1] + y[1:]) * a).sum() / (6 * A)
    return float(cy), float(cx)

geo = json.load(open(GEO, encoding="utf-8"))
geo_puno = {"type": "FeatureCollection",
            "features": [f for f in geo["features"]
                         if f["properties"]["IDDPTO"] == "21" and f["geometry"] is not None]}

# Amantaní (isla del Titicaca) viene sin geometría en el GeoJSON: se usa un centroide manual.
CENTROIDE_MANUAL = {"210103": (-15.66, -69.72)}

dim = []
for f in geo["features"]:
    p = f["properties"]
    if p["IDDPTO"] != "21":
        continue
    lat, lon = CENTROIDE_MANUAL.get(p["IDDIST"]) or centroide(f["geometry"])
    dim.append({"geo_id": p["IDDIST"], "prov_n": norm(p["NOMBPROV"]), "dist_n": norm(p["NOMBDIST"]),
                "nombre_geo": p["NOMBDIST"], "lat": lat, "lon": lon, "area_km2": p["AREA_MINAM"] / 100})
dim = pd.DataFrame(dim)
print("Distritos de Puno en el GeoJSON:", len(dim))
dim.head()

Distritos de Puno en el GeoJSON: 109


,geo_id,prov_n,dist_n,nombre_geo,lat,lon,area_km2
0,210502,EL COLLAO,CAPAZO,CAPAZO,-17.11,-69.70,"1,048.68"
1,210405,CHUCUITO,PISACOMA,PISACOMA,-16.93,-69.45,963.84
2,210404,CHUCUITO,KELLUYO,KELLUYO,-16.72,-69.25,496.20
3,210402,CHUCUITO,DESAGUADERO,DESAGUADERO,-16.62,-69.08,176.15
4,210504,EL COLLAO,SANTA ROSA,SANTA ROSA,-16.83,-69.86,"2,712.13"


In [14]:
ALIAS = {("EL COLLAO", "CAPASO"): ("EL COLLAO", "CAPAZO"),
         ("SANDIA", "PUTINA PUNCU"): ("SANDIA", "SAN PEDRO DE PUTINA PUNCO"),
         ("SAN ROMAN", "SAN MIGUEL"): ("SAN ROMAN", "JULIACA")}   # sin polígono propio
UBIGEO_ESPECIAL = {("SAN ROMAN", "SAN MIGUEL"): "211105"}

pares = df[["provincia", "distrito"]].drop_duplicates().copy()
pares["prov_n"], pares["dist_n"] = zip(*[ALIAS.get((norm(p), norm(d)), (norm(p), norm(d)))
                                         for p, d in zip(pares["provincia"], pares["distrito"])])
pares = pares.merge(dim, on=["prov_n", "dist_n"], how="left")
pares["ubigeo"] = [UBIGEO_ESPECIAL.get((p, d), g) for p, d, g in zip(pares["provincia"], pares["distrito"], pares["geo_id"])]
print("Distritos sin geolocalizar:", pares["geo_id"].isna().sum())
pares.loc[pares["provincia"].eq("SAN ROMAN") | pares["distrito"].eq("AMANTANI")]

Distritos sin geolocalizar: 0


,provincia,distrito,prov_n,dist_n,geo_id,nombre_geo,lat,lon,area_km2,ubigeo
22,PUNO,AMANTANI,PUNO,AMANTANI,210103,AMANTANI,-15.66,-69.72,13.94,210103
34,SAN ROMAN,CABANILLAS,SAN ROMAN,CABANILLAS,211103,CABANILLAS,-15.85,-70.62,"1,274.34",211103
35,SAN ROMAN,JULIACA,SAN ROMAN,JULIACA,211101,JULIACA,-15.45,-70.19,533.99,211101
36,SAN ROMAN,SAN MIGUEL,SAN ROMAN,JULIACA,211101,JULIACA,-15.45,-70.19,533.99,211105
62,SAN ROMAN,CARACOTO,SAN ROMAN,CARACOTO,211104,CARACOTO,-15.56,-70.09,282.31,211104
86,SAN ROMAN,CABANA,SAN ROMAN,CABANA,211102,CABANA,-15.65,-70.28,193.94,211102


In [15]:
df = df.merge(pares.drop(columns=["prov_n", "dist_n"]), on=["provincia", "distrito"], how="left")
df["ubigeo_corregido"] = df["ubigeo_origen"] != df["ubigeo"]
print(f"Registros con ubigeo corregido: {df['ubigeo_corregido'].sum():,} ({df['ubigeo_corregido'].mean():.1%})")

Registros con ubigeo corregido: 11,004 (39.5%)


## 6. Variables derivadas (transformación)
| Variable | Fórmula | Interpretación |
|---|---|---|
| `kwh_cliente` | consumo_kwh / clientes | Consumo medio mensual por suministro |
| `soles_cliente` | monto_soles / clientes | Facturación media por suministro |
| `precio_kwh` | monto_soles / consumo_kwh | Precio medio efectivo (S/ por kWh); NaN si no hay kWh |
| `sin_consumo` | consumo_kwh == 0 | Suministro sin consumo registrado (cargo fijo) |
| `rango_consumo` | cortes de `kwh_cliente` | 0-30, 31-100, 101-300, >300 kWh (el umbral de 100 kWh es el del FOSE) |

In [16]:
df["kwh_cliente"] = df["consumo_kwh"] / df["clientes"]
df["soles_cliente"] = df["monto_soles"] / df["clientes"]
df["precio_kwh"] = np.where(df["consumo_kwh"] > 0, df["monto_soles"] / df["consumo_kwh"], np.nan)
df["sin_consumo"] = df["consumo_kwh"].eq(0)
df["sin_facturacion"] = df["monto_soles"].eq(0)
df["rango_consumo"] = pd.cut(df["kwh_cliente"], [-0.1, 30, 100, 300, np.inf],
                             labels=["0-30 kWh", "31-100 kWh", "101-300 kWh", ">300 kWh"])
df[["kwh_cliente", "soles_cliente", "precio_kwh"]].describe()

,kwh_cliente,soles_cliente,precio_kwh
count,"27,875.00","27,875.00","20,871.00"
mean,715.68,"3,127.22",1.77
std,"4,242.91","33,268.75",24.64
min,0.00,"-1,860.37",-1.70
25%,0.00,18.10,0.83
50%,17.76,74.00,1.13
75%,230.89,778.95,1.43
max,"83,054.56","828,526.60","2,833.08"


## 7. Vista rápida de la ubicación de los registros

In [17]:
pts = (df.groupby(["ubigeo", "distrito", "provincia", "lat", "lon"], as_index=False)
         .agg(clientes=("clientes", "sum"), consumo_kwh=("consumo_kwh", "sum")))
fig = px.scatter_map(pts, lat="lat", lon="lon", size="clientes", color="provincia",
                     hover_name="distrito", hover_data={"clientes": ":,", "consumo_kwh": ":,.0f"},
                     zoom=6, height=550, map_style="carto-positron",
                     title="Clientes de Electro Puno por distrito (centroides)")
fig.show()

## 8. Almacenamiento
- `consumo_limpio.parquet` / `.csv`: tabla limpia a nivel de registro.
- `puno_distritos.geojson`: GeoJSON filtrado a Puno (más liviano para el dashboard).
- `dim_distritos.csv`: dimensión geográfica (ubigeo, centroide, área).

In [18]:
cols = ["periodo", "periodo_cod", "anio", "mes", "fecha_reporte", "departamento", "provincia", "distrito",
        "ubigeo", "geo_id", "ubigeo_origen", "ubigeo_corregido", "lat", "lon", "area_km2",
        "tarifa", "nivel_tension", "grupo_tarifa", "estado", "clientes", "consumo_kwh", "monto_soles",
        "kwh_cliente", "soles_cliente", "precio_kwh", "sin_consumo", "sin_facturacion", "rango_consumo", "archivo"]
limpio = df[cols].sort_values(["periodo", "provincia", "distrito", "tarifa"]).reset_index(drop=True)

limpio.to_parquet(OUT / "consumo_limpio.parquet", index=False)
limpio.to_csv(OUT / "consumo_limpio.csv", index=False, encoding="utf-8-sig")
json.dump(geo_puno, open(OUT / "puno_distritos.geojson", "w", encoding="utf-8"))
pares.drop(columns=["prov_n", "dist_n"]).to_csv(OUT / "dim_distritos.csv", index=False, encoding="utf-8-sig")

print("Guardado:", [p.name for p in OUT.iterdir()])
print("Registros:", len(limpio), "| Periodos:", limpio["periodo_cod"].nunique(),
      "| Distritos:", limpio["ubigeo"].nunique(), "| Clientes:", f"{limpio['clientes'].sum():,}")

Guardado: ['ajuste_provincia.csv', 'comparacion_distribuciones.csv', 'consumo_final.parquet', 'consumo_limpio.csv', 'consumo_limpio.parquet', 'dim_distritos.csv', 'distribuciones_resumen.json', 'indicadores_distrito.csv', 'indicadores_provincia.csv', 'kpis_gerencia.csv', 'puno_distritos.geojson', 'raw_archivos.csv', 'raw_unificado.parquet', 'registros_anomalos.csv']
Registros: 27875 | Periodos: 42 | Distritos: 109 | Clientes: 14,674,499
